In [0]:
!pip install python-dotenv

In [0]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, col, count
from dotenv import load_dotenv

load_dotenv()

In [0]:
# 3. Credentials
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# 4. Path Definition
bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

print(f"✅ Connected to Bronze Layer at: {bronze_path}")

In [0]:
# Load all raw JSON files
# Using recursiveFileLookup to find files in date-partitioned folders
raw_df = spark.read.option("recursiveFileLookup", "true").json(bronze_path)

# Add the file path so we can see which folder/file the data came from
raw_df_with_source = raw_df.withColumn("source_file", input_file_name())

print(f"📊 Total Raw Records Found: {raw_df_with_source.count()}")
raw_df_with_source.select("source", "endpoint_type", "source_file").show(10, truncate=False)

In [0]:
print("🔍 Inspecting Raw JSON Structure:")
raw_df_with_source.printSchema()

# Check for specific endpoints found in Bronze
raw_df_with_source.groupBy("source", "endpoint_type").count().show()

In [0]:
# Check if the payload is arriving empty
null_payload_count = raw_df_with_source.filter(col("payload").isNull()).count()

if null_payload_count > 0:
    print(f"⚠️ WARNING: Found {null_payload_count} records with empty payloads!")
else:
    print("✅ Success: All payloads contain data.")

# Sample view of the nested payload
print("👀 Sample Payload Content:")
raw_df_with_source.select("endpoint_type", "payload").limit(3).show(truncate=80)